In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")


# COVID-19 Round 19

In [9]:
def pull_surveillance_data():
    #flu surveillance data
    
    url = f"https://raw.githubusercontent.com/CDCgov/covid19-forecast-hub/refs/heads/main/target-data/covid-hospital-admissions.csv"
    return pd.read_csv(url, dtype={'location':str})


In [10]:
models = ['CFA-Scenarios', 'JHU_UNC-flepiMoP', 'LEMMA-EnsembleDTWS', 'MOBS_NEU-GLEAM_COVID', 'PSI-M_CoV_2025',
            'UIUC-IMMCYC', 'UNCC-Hierbin', 'UT-ImmunoSEIRS', 'UVA-adaptive']
dates = '2025-04-27'
rd = 19

modelmap = {'MOBS_NEU-GLEAM_COVID':'Model A', 'CFA-Scenarios':'Model B', 'JHU_UNC-flepiMoP': 'Model C', 
            'PSI-M_CoV_2025':'Model D', 'UIUC-IMMCYC':'Model E', 'UNCC-Hierbin':'Model F', 
            'UT-ImmunoSEIRS':'Model G', 'UVA-adaptive':'Model H'}

observations = pull_surveillance_data()

In [6]:
es = pd.read_pickle("../coviddat/energyscore_individual_rd19_covidhosp.pkl")
wisdf = pd.read_pickle("../coviddat/wis_trajectory_indiv_covid_rd19_hosp.pkl")
    
allscores = es.merge(wisdf, on=['location', 'Label','Model','abbreviation', 'location_name', 'population', 'target']).dropna()
allscores['modelmap'] = allscores['Model'].apply(lambda x: modelmap[x])
allscores = allscores.sort_values(by='modelmap')



In [7]:
predictionsall = pd.DataFrame()
for model in models:
    df = pd.read_parquet(f'../coviddat/{model}_rd4_hosp.pq')
    print(model, len(df.location.unique())) 
    
    predictionsall = pd.concat([predictionsall, df])


CFA-Scenarios 51
JHU_UNC-flepiMoP 52
LEMMA-EnsembleDTWS 52
MOBS_NEU-GLEAM_COVID 52
PSI-M_CoV_2025 52
UIUC-IMMCYC 52
UNCC-Hierbin 52
UT-ImmunoSEIRS 51
UVA-adaptive 52


NameError: name 'observations' is not defined

In [11]:
# filter by trajectories and only look at age group with all ages combined
predictions_traj = predictionsall[(predictionsall.output_type == 'sample') & \
                                   (predictionsall.age_group == '0-130')]
# filter by dates with data
predictions_traj = predictions_traj[predictions_traj.target_end_date <= pd.to_datetime(observations.date.max())]

In [23]:
predictions_traj['run_id'] = predictions_traj.groupby(['stochastic_run', 'run_grouping']).ngroup()
predictions_traj['Label'] = predictions_traj['scenario_id'].apply(lambda x: 'Scenario ' + x[0])
predictions_traj['modelmap'] = predictions_traj['Model'].apply(lambda x: modelmap[x])

In [25]:
predictions_traj

,origin_date,scenario_id,location,target,horizon,age_group,output_type,output_type_id,run_grouping,stochastic_run,value,Model,target_end_date,run_id,Label,modelmap
0,2025-04-27,A-2025-04-01,02,inc hosp,1.0,0-130,sample,NaN,1.0,1.0,17.0,CFA-Scenarios,2025-05-03,1,Scenario A,Model B
3,2025-04-27,A-2025-04-01,02,inc hosp,1.0,0-130,sample,NaN,1.0,2.0,14.0,CFA-Scenarios,2025-05-03,300,Scenario A,Model B
6,2025-04-27,A-2025-04-01,02,inc hosp,1.0,0-130,sample,NaN,1.0,3.0,16.0,CFA-Scenarios,2025-05-03,302,Scenario A,Model B
9,2025-04-27,A-2025-04-01,02,inc hosp,1.0,0-130,sample,NaN,1.0,4.0,5.0,CFA-Scenarios,2025-05-03,304,Scenario A,Model B
12,2025-04-27,A-2025-04-01,02,inc hosp,1.0,0-130,sample,NaN,1.0,5.0,14.0,CFA-Scenarios,2025-05-03,306,Scenario A,Model B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4509175,2025-04-27,E-2025-04-01,US,inc hosp,47.0,0-130,sample,NaN,96.0,1.0,10244.1,UVA-adaptive,2026-03-21,96,Scenario E,Model H
4509176,2025-04-27,E-2025-04-01,US,inc hosp,47.0,0-130,sample,NaN,97.0,1.0,9452.4,UVA-adaptive,2026-03-21,97,Scenario E,Model H
4509177,2025-04-27,E-2025-04-01,US,inc hosp,47.0,0-130,sample,NaN,98.0,1.0,10694.5,UVA-adaptive,2026-03-21,98,Scenario E,Model H
4509178,2025-04-27,E-2025-04-01,US,inc hosp,47.0,0-130,sample,NaN,99.0,1.0,8568.4,UVA-adaptive,2026-03-21,99,Scenario E,Model H


In [34]:

# =============================================================================
# CONFIG — adjust column names to match your data
# =============================================================================

# Trajectory dataframe columns
COL_MODEL = "modelmap"
COL_LOCATION = "location"
COL_SCENARIO = "Label"
COL_SAMPLE = "run_id"
COL_TIME = "target_end_date"
COL_VALUE = "value"
COL_ES = "energyscore"
COL_WIS = "WIS"
LOWER_IS_BETTER = True

# Group columns (must match between trajectory df and score df)
GROUP_COLS = [COL_LOCATION, COL_SCENARIO, COL_MODEL]

# -- Load your data --
df_traj = predictions_traj.copy()


df = allscores.copy()
GROUP_COLS1 = [COL_LOCATION, COL_SCENARIO]

# Deduplicate in case of any exact duplicates
df = df.drop_duplicates(subset=GROUP_COLS1 + [COL_MODEL])

# Step 1: Rank models within each (location, scenario)
df["rank_es"] = df.groupby(GROUP_COLS1)[COL_ES].rank(
    ascending=LOWER_IS_BETTER, method="min"
).astype(int)
df["rank_wis"] = df.groupby(GROUP_COLS1)[COL_WIS].rank(
    ascending=LOWER_IS_BETTER, method="min"
).astype(int)

df["rank_diff"] = (df["rank_es"] - df["rank_wis"]).abs()
df["ranks_agree"] = df["rank_es"] == df["rank_wis"]

df_scores = df.copy()

# =============================================================================
# STEP 1: Compute distributional features per (model, location, scenario)
# =============================================================================

def compute_trajectory_features(group):
    """
    Given trajectories for one (model, location, scenario),
    compute summary features of the forecast distribution.
    
    group: DataFrame with columns [sample_id, time, value]
    """
    # Pivot to (sample_id x time) matrix
    traj_matrix = group.pivot_table(
        index=COL_SAMPLE, columns=COL_TIME, values=COL_VALUE
    )
    n_samples, n_times = traj_matrix.shape

    features = {}

    # --- Marginal features (averaged across timepoints) ---

    # Mean and variance of pointwise means
    pointwise_mean = traj_matrix.mean(axis=0)
    pointwise_var = traj_matrix.var(axis=0)
    features["mean_forecast"] = pointwise_mean.mean()
    features["mean_variance"] = pointwise_var.mean()
    features["max_variance"] = pointwise_var.max()

    # Sharpness: average IQR across timepoints
    q25 = traj_matrix.quantile(0.25, axis=0)
    q75 = traj_matrix.quantile(0.75, axis=0)
    q025 = traj_matrix.quantile(0.025, axis=0)
    q975 = traj_matrix.quantile(0.975, axis=0)
    features["mean_iqr"] = (q75 - q25).mean()
    features["mean_90_range"] = (q975 - q025).mean()

    # Tail heaviness: ratio of 95% range to IQR
    iqr = (q75 - q25).replace(0, np.nan)
    tail_ratio = (q975 - q025) / iqr
    features["mean_tail_ratio"] = tail_ratio.mean()

    # Skewness and kurtosis of marginal distributions (averaged)
    pointwise_skew = traj_matrix.apply(lambda col: col.skew(), axis=0)
    pointwise_kurt = traj_matrix.apply(lambda col: col.kurtosis(), axis=0)
    features["mean_skewness"] = pointwise_skew.mean()
    features["mean_kurtosis"] = pointwise_kurt.mean()

    # --- Bimodality detection ---
    # Dip in the middle of the distribution at peak variance timepoint
    peak_var_time = pointwise_var.idxmax()
    peak_samples = traj_matrix[peak_var_time].dropna().values
    # Hartigan's dip test approximation: use bimodality coefficient
    n = len(peak_samples)
    skew = stats.skew(peak_samples)
    kurt = stats.kurtosis(peak_samples)  # excess kurtosis
    # Bimodality coefficient: BC > 0.555 suggests bimodality
    features["bimodality_coeff"] = (skew**2 + 1) / (kurt + 3 * (n-1)**2 / ((n-2)*(n-3)))

    # --- Temporal features ---

    # Trajectory spread over time: std of pointwise variance
    features["variance_variability"] = pointwise_var.std()

    # Inter-trajectory correlation: how similar are trajectories to each other?
    # Sample a subset for speed
    if n_samples > 50:
        sample_idx = np.random.choice(traj_matrix.index, 50, replace=False)
        traj_sub = traj_matrix.loc[sample_idx]
    else:
        traj_sub = traj_matrix
    corr_matrix = traj_sub.T.corr()
    # Mean pairwise correlation (upper triangle)
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
    )
    features["mean_traj_correlation"] = upper_tri.stack().mean()

    # Peak timing diversity: std of time-of-peak across trajectories
    peak_times = traj_matrix.idxmax(axis=1)
    features["peak_time_std"] = peak_times.std()

    # Peak magnitude diversity
    peak_vals = traj_matrix.max(axis=1)
    features["peak_magnitude_cv"] = peak_vals.std() / (peak_vals.mean() + 1e-8)

    return pd.Series(features)


print("Computing distributional features (this may take a few minutes)...")
df_features = df_traj.groupby(GROUP_COLS).apply(
    compute_trajectory_features
).reset_index()
print(f"Computed features for {len(df_features)} (model, location, scenario) groups")

# =============================================================================
# STEP 2: Merge with score/rank data
# =============================================================================

# df_scores should already have rank_es and rank_wis from previous analysis
df_merged = df_features.merge(
    df_scores[GROUP_COLS + ["rank_es", "rank_wis", "energyscore", "WIS"]],
    on=GROUP_COLS,
    how="inner"
)

# Target variable: rank difference (positive = WIS ranks model higher than ES)
df_merged["rank_diff"] = df_merged["rank_wis"] - df_merged["rank_es"]
df_merged["abs_rank_diff"] = df_merged["rank_diff"].abs()

print(f"Merged dataset: {len(df_merged)} rows")

# =============================================================================
# STEP 3: OLS Regression — what predicts rank disagreement?
# =============================================================================


feature_cols = [
    "mean_forecast", "mean_variance", "max_variance",
    "mean_iqr", "mean_90_range", "mean_tail_ratio",
    "mean_skewness", "mean_kurtosis", "bimodality_coeff",
    "variance_variability", "mean_traj_correlation",
    "peak_time_std", "peak_magnitude_cv"
]

# Drop rows with NaN/inf in features
df_reg = df_merged[GROUP_COLS + feature_cols + ["rank_diff", "abs_rank_diff"]].copy()
df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna()

print(f"\n{'='*65}")
print("OLS REGRESSION: Distributional Features → Rank Difference (WIS - ES)")
print(f"{'='*65}")
print(f"N = {len(df_reg)}")
print(f"\nPositive rank_diff = WIS ranks model higher (better) than ES")
print(f"Negative rank_diff = ES ranks model higher (better) than WIS\n")

# Force numeric types and check for problems
X = df_reg[feature_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce")
df_reg["rank_diff"] = pd.to_numeric(df_reg["rank_diff"], errors="coerce")

# Drop any rows that became NaN after coercion
valid = X.notna().all(axis=1) & df_reg["rank_diff"].notna()
X = X[valid]
df_reg = df_reg[valid]

print(f"Rows after cleaning: {len(df_reg)}")
print(f"Feature dtypes:\n{X.dtypes}\n")

# Standardize features for comparable coefficients
X_means = X.mean()
X_stds = X.std()
X_standardized = (X - X_means) / X_stds
X_standardized = sm.add_constant(X_standardized)

y = df_reg["rank_diff"]

model = sm.OLS(y, X_standardized).fit(cov_type="HC3")  # robust SEs
print(model.summary())

# =============================================================================
# STEP 4: Formatted coefficient table for the paper
# =============================================================================

print(f"\n{'='*65}")
print("COEFFICIENT TABLE (standardized, robust SEs)")
print(f"{'='*65}")

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coef": model.params[1:].values,
    "se": model.bse[1:].values,
    "t": model.tvalues[1:].values,
    "p": model.pvalues[1:].values,
}).sort_values("p")

coef_table["sig"] = coef_table["p"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
)

print(coef_table.to_string(index=False, float_format="%.4f"))

print(f"\nR² = {model.rsquared:.4f}")
print(f"Adj R² = {model.rsquared_adj:.4f}")

# =============================================================================
# STEP 5: Interpretation helper
# =============================================================================

print(f"\n{'='*65}")
print("INTERPRETATION")
print(f"{'='*65}")

sig_features = coef_table[coef_table["p"] < 0.05]
if len(sig_features) > 0:
    print("\nSignificant predictors of rank disagreement (p < 0.05):\n")
    for _, row in sig_features.iterrows():
        direction = "WIS ranks higher" if row["coef"] > 0 else "ES ranks higher"
        print(f"  {row['feature']}: 1 SD increase → {row['coef']:.3f} rank shift "
              f"({direction}) [p={row['p']:.4f}]")
else:
    print("\nNo features significant at p < 0.05")

print(f"\nCorrelation of features with rank_diff:")
corrs = df_reg[feature_cols + ["rank_diff"]].corr()["rank_diff"].drop("rank_diff")
print(corrs.sort_values(key=abs, ascending=False).round(4).to_string())

Computing distributional features (this may take a few minutes)...
Computed features for 2070 (model, location, scenario) groups
Merged dataset: 1790 rows

OLS REGRESSION: Distributional Features → Rank Difference (WIS - ES)
N = 1790

Positive rank_diff = WIS ranks model higher (better) than ES
Negative rank_diff = ES ranks model higher (better) than WIS

Rows after cleaning: 1790
Feature dtypes:
mean_forecast            float64
mean_variance            float64
max_variance             float64
mean_iqr                 float64
mean_90_range            float64
mean_tail_ratio          float64
mean_skewness            float64
mean_kurtosis            float64
bimodality_coeff         float64
variance_variability     float64
mean_traj_correlation    float64
peak_time_std              int64
peak_magnitude_cv        float64
dtype: object

                            OLS Regression Results                            
Dep. Variable:              rank_diff   R-squared:                       0.11

In [35]:
def pull_surveillance_data():
    #flu surveillance data
    
    url = f"https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/target-data/target-hospital-admissions.csv"
    return pd.read_csv(url, dtype={'location':str})


In [40]:
observations = pull_surveillance_data()

models = ['ACCIDDA-FlepiMoP', 'CADPH-FluCAT', 'MOBS_NEU-GLEAM_FLU',  'NIH-Flu_TS', 'NotreDame-FRED', 'PSI-M2', 
          'SigSci-SWIFT', 'USC-SIkJalpha', 'UT-ImmunoSEIRS', 'UVA-EscapeFlu', 'UVA-FluXSim']
dates = '2024-08-11'
rd = 5

modelmap = {'MOBS_NEU-GLEAM_FLU':'Model A', 'NIH-Flu_TS': 'Model B','PSI-M2':'Model C', 'NotreDame-FRED':'Model D',
            'USC-SIkJalpha':'Model E', 'UT-ImmunoSEIRS':'Model F', 'ACCIDDA-FlepiMoP': 'Model G',
           'SigSci-SWIFT':'Model H', 'UVA-EscapeFlu':'Model I', 'UVA-FluXSim':'Model J'}


In [37]:
predictionsall = pd.DataFrame()
for model in models:
    df = pd.read_parquet(f'../fludat/{model}_rd{rd}.pq')
    print(model, len(df.location.unique())) 
    
    predictionsall = pd.concat([predictionsall, df])

ACCIDDA-FlepiMoP 52
CADPH-FluCAT 1
MOBS_NEU-GLEAM_FLU 52
NIH-Flu_TS 39
NotreDame-FRED 52
PSI-M2 52
SigSci-SWIFT 51
USC-SIkJalpha 57
UT-ImmunoSEIRS 51
UVA-EscapeFlu 51
UVA-FluXSim 52


In [38]:
singleloc_models = ['CADPH-FluCAT']

predictionsall = predictionsall[~predictionsall['Model'].isin(singleloc_models)]

# filter by trajectories and only look at age group with all ages combined
predictions_traj = predictionsall[(predictionsall.output_type == 'sample') & \
                                   (predictionsall.age_group == '0-130')]
# filter by dates with data
predictions_traj = predictions_traj[predictions_traj.target_end_date <= pd.to_datetime('2025-06-08')]

In [41]:
es = pd.read_pickle("../fludat/energyscore_individual_rd5_fluhosp.pkl")

wisdf = pd.read_pickle("../fludat/wis_trajectory_indiv_flu_rd5_hosp.pkl")
    
allscores = es.merge(wisdf, on=['location', 'Label','Model', 'target', 'abbreviation','location_name','population']).dropna()
allscores['modelmap'] = allscores['Model'].apply(lambda x: modelmap[x])
allscores = allscores.sort_values(by='modelmap')


In [42]:
predictions_traj['run_id'] = predictions_traj.groupby(['stochastic_run', 'run_grouping']).ngroup()
predictions_traj['Label'] = predictions_traj['scenario_id'].apply(lambda x: 'Scenario ' + x[0])
predictions_traj['modelmap'] = predictions_traj['Model'].apply(lambda x: modelmap[x])

In [43]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG — adjust column names to match your data
# =============================================================================

# Trajectory dataframe columns
COL_MODEL = "modelmap"
COL_LOCATION = "location"
COL_SCENARIO = "Label"
COL_SAMPLE = "run_id"
COL_TIME = "target_end_date"
COL_VALUE = "value"
COL_ES = "energyscore"
COL_WIS = "WIS"
LOWER_IS_BETTER = True

# Group columns (must match between trajectory df and score df)
GROUP_COLS = [COL_LOCATION, COL_SCENARIO, COL_MODEL]

# -- Load your data --
df_traj = predictions_traj.copy()

df = allscores.copy()
GROUP_COLS1 = [COL_LOCATION, COL_SCENARIO]

# Deduplicate in case of any exact duplicates
df = df.drop_duplicates(subset=GROUP_COLS1 + [COL_MODEL])

# Step 1: Rank models within each (location, scenario)
df["rank_es"] = df.groupby(GROUP_COLS1)[COL_ES].rank(
    ascending=LOWER_IS_BETTER, method="min"
).astype(int)
df["rank_wis"] = df.groupby(GROUP_COLS1)[COL_WIS].rank(
    ascending=LOWER_IS_BETTER, method="min"
).astype(int)

df["rank_diff"] = (df["rank_es"] - df["rank_wis"]).abs()
df["ranks_agree"] = df["rank_es"] == df["rank_wis"]

df_scores = df.copy()

# =============================================================================
# STEP 1: Compute distributional features per (model, location, scenario)
# =============================================================================

def compute_trajectory_features(group):
    """
    Given trajectories for one (model, location, scenario),
    compute summary features of the forecast distribution.
    
    group: DataFrame with columns [sample_id, time, value]
    """
    # Pivot to (sample_id x time) matrix
    traj_matrix = group.pivot_table(
        index=COL_SAMPLE, columns=COL_TIME, values=COL_VALUE
    )
    n_samples, n_times = traj_matrix.shape

    features = {}

    # --- Marginal features (averaged across timepoints) ---

    # Mean and variance of pointwise means
    pointwise_mean = traj_matrix.mean(axis=0)
    pointwise_var = traj_matrix.var(axis=0)
    features["mean_forecast"] = pointwise_mean.mean()
    features["mean_variance"] = pointwise_var.mean()
    features["max_variance"] = pointwise_var.max()

    # Sharpness: average IQR across timepoints
    q25 = traj_matrix.quantile(0.25, axis=0)
    q75 = traj_matrix.quantile(0.75, axis=0)
    q025 = traj_matrix.quantile(0.025, axis=0)
    q975 = traj_matrix.quantile(0.975, axis=0)
    features["mean_iqr"] = (q75 - q25).mean()
    features["mean_90_range"] = (q975 - q025).mean()

    # Tail heaviness: ratio of 95% range to IQR
    iqr = (q75 - q25).replace(0, np.nan)
    tail_ratio = (q975 - q025) / iqr
    features["mean_tail_ratio"] = tail_ratio.mean()

    # Skewness and kurtosis of marginal distributions (averaged)
    pointwise_skew = traj_matrix.apply(lambda col: col.skew(), axis=0)
    pointwise_kurt = traj_matrix.apply(lambda col: col.kurtosis(), axis=0)
    features["mean_skewness"] = pointwise_skew.mean()
    features["mean_kurtosis"] = pointwise_kurt.mean()

    # --- Bimodality detection ---
    # Dip in the middle of the distribution at peak variance timepoint
    peak_var_time = pointwise_var.idxmax()
    peak_samples = traj_matrix[peak_var_time].dropna().values
    # Hartigan's dip test approximation: use bimodality coefficient
    n = len(peak_samples)
    skew = stats.skew(peak_samples)
    kurt = stats.kurtosis(peak_samples)  # excess kurtosis
    # Bimodality coefficient: BC > 0.555 suggests bimodality
    features["bimodality_coeff"] = (skew**2 + 1) / (kurt + 3 * (n-1)**2 / ((n-2)*(n-3)))

    # --- Temporal features ---

    # Trajectory spread over time: std of pointwise variance
    features["variance_variability"] = pointwise_var.std()

    # Inter-trajectory correlation: how similar are trajectories to each other?
    # Sample a subset for speed
    if n_samples > 50:
        sample_idx = np.random.choice(traj_matrix.index, 50, replace=False)
        traj_sub = traj_matrix.loc[sample_idx]
    else:
        traj_sub = traj_matrix
    corr_matrix = traj_sub.T.corr()
    # Mean pairwise correlation (upper triangle)
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
    )
    features["mean_traj_correlation"] = upper_tri.stack().mean()

    # Peak timing diversity: std of time-of-peak across trajectories
    peak_times = traj_matrix.idxmax(axis=1)
    features["peak_time_std"] = peak_times.std()

    # Peak magnitude diversity
    peak_vals = traj_matrix.max(axis=1)
    features["peak_magnitude_cv"] = peak_vals.std() / (peak_vals.mean() + 1e-8)

    return pd.Series(features)


print("Computing distributional features (this may take a few minutes)...")
df_features = df_traj.groupby(GROUP_COLS).apply(
    compute_trajectory_features
).reset_index()
print(f"Computed features for {len(df_features)} (model, location, scenario) groups")

# =============================================================================
# STEP 2: Merge with score/rank data
# =============================================================================

# df_scores should already have rank_es and rank_wis from previous analysis
df_merged = df_features.merge(
    df_scores[GROUP_COLS + ["rank_es", "rank_wis", "energyscore", "WIS"]],
    on=GROUP_COLS,
    how="inner"
)

# Target variable: rank difference (positive = WIS ranks model higher than ES)
df_merged["rank_diff"] = df_merged["rank_wis"] - df_merged["rank_es"]
df_merged["abs_rank_diff"] = df_merged["rank_diff"].abs()

print(f"Merged dataset: {len(df_merged)} rows")

# =============================================================================
# STEP 3: OLS Regression — what predicts rank disagreement?
# =============================================================================

feature_cols = [
    "mean_forecast", "mean_variance", "max_variance",
    "mean_iqr", "mean_90_range", "mean_tail_ratio",
    "mean_skewness", "mean_kurtosis", "bimodality_coeff",
    "variance_variability", "mean_traj_correlation",
    "peak_time_std", "peak_magnitude_cv"
]

# Drop rows with NaN/inf in features
df_reg = df_merged[GROUP_COLS + feature_cols + ["rank_diff", "abs_rank_diff"]].copy()
df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna()

print(f"\n{'='*65}")
print("OLS REGRESSION: Distributional Features → Rank Difference (WIS - ES)")
print(f"{'='*65}")
print(f"N = {len(df_reg)}")
print(f"\nPositive rank_diff = WIS ranks model higher (better) than ES")
print(f"Negative rank_diff = ES ranks model higher (better) than WIS\n")

# Force numeric types and check for problems
X = df_reg[feature_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce")
df_reg["rank_diff"] = pd.to_numeric(df_reg["rank_diff"], errors="coerce")

# Drop any rows that became NaN after coercion
valid = X.notna().all(axis=1) & df_reg["rank_diff"].notna()
X = X[valid]
df_reg = df_reg[valid]

print(f"Rows after cleaning: {len(df_reg)}")
print(f"Feature dtypes:\n{X.dtypes}\n")

# Standardize features for comparable coefficients
X_means = X.mean()
X_stds = X.std()
X_standardized = (X - X_means) / X_stds
X_standardized = sm.add_constant(X_standardized)

y = df_reg["rank_diff"]

model = sm.OLS(y, X_standardized).fit(cov_type="HC3")  # robust SEs
print(model.summary())

# =============================================================================
# STEP 4: Formatted coefficient table for the paper
# =============================================================================

print(f"\n{'='*65}")
print("COEFFICIENT TABLE (standardized, robust SEs)")
print(f"{'='*65}")

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coef": model.params[1:].values,
    "se": model.bse[1:].values,
    "t": model.tvalues[1:].values,
    "p": model.pvalues[1:].values,
}).sort_values("p")

coef_table["sig"] = coef_table["p"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
)

print(coef_table.to_string(index=False, float_format="%.4f"))

print(f"\nR² = {model.rsquared:.4f}")
print(f"Adj R² = {model.rsquared_adj:.4f}")

# =============================================================================
# STEP 5: Interpretation helper
# =============================================================================

print(f"\n{'='*65}")
print("INTERPRETATION")
print(f"{'='*65}")

sig_features = coef_table[coef_table["p"] < 0.05]
if len(sig_features) > 0:
    print("\nSignificant predictors of rank disagreement (p < 0.05):\n")
    for _, row in sig_features.iterrows():
        direction = "WIS ranks higher" if row["coef"] > 0 else "ES ranks higher"
        print(f"  {row['feature']}: 1 SD increase → {row['coef']:.3f} rank shift "
              f"({direction}) [p={row['p']:.4f}]")
else:
    print("\nNo features significant at p < 0.05")

print(f"\nCorrelation of features with rank_diff:")
corrs = df_reg[feature_cols + ["rank_diff"]].corr()["rank_diff"].drop("rank_diff")
print(corrs.sort_values(key=abs, ascending=False).round(4).to_string())

Computing distributional features (this may take a few minutes)...
Computed features for 3054 (model, location, scenario) groups
Merged dataset: 2616 rows

OLS REGRESSION: Distributional Features → Rank Difference (WIS - ES)
N = 2609

Positive rank_diff = WIS ranks model higher (better) than ES
Negative rank_diff = ES ranks model higher (better) than WIS

Rows after cleaning: 2609
Feature dtypes:
mean_forecast            float64
mean_variance            float64
max_variance             float64
mean_iqr                 float64
mean_90_range            float64
mean_tail_ratio          float64
mean_skewness            float64
mean_kurtosis            float64
bimodality_coeff         float64
variance_variability     float64
mean_traj_correlation    float64
peak_time_std              int64
peak_magnitude_cv        float64
dtype: object

                            OLS Regression Results                            
Dep. Variable:              rank_diff   R-squared:                       0.05